# 53 — Skill Gap Analysis
**Goal:** Identify missing skills and generate upskilling recommendations.

Scoring says *how far* a candidate is from the job; this chapter says *what exactly is missing*. `SkillGapAnalyzer` matches the resume text against a curated **skills taxonomy** (six categories), compares what was found against the core and advanced skill lists of a target **career path**, and returns a coverage ratio plus explicit recommendations.

**Why it matters for resumes / ATS:** gap analysis is the coaching layer of the ATS. A recruiter can tell a candidate "you matched 50%" — but only a per-skill breakdown can say "you're missing Kubernetes, MLOps, and CI/CD; learn those first." The same machinery powers candidate coaching, hiring-manager debriefs, and internal upskilling programs.

## 1. Finding Skill Gaps

`analyze_gaps(resume_text, target_role)` works in three steps: **scan** the resume for taxonomy skills (case-insensitive substring match), **look up** the target role's `core`/`advanced` requirement lists, and **compare** to produce `coverage`, met/gap lists, and recommendations.

**What the code does:**
- `skills_taxonomy` — 6 categories (programming, ml_dl, nlp, data, cloud_devops, databases) with ~30 curated skills; only skills in this list can ever be detected.
- `career_paths` — 3 roles (`data_scientist`, `ml_engineer`, `nlp_engineer`), each with a `core` and an `advanced` list; an unknown role returns `{"error": ...}` instead of crashing.
- `coverage` — `(core_found + adv_found) / (core_total + adv_total)`, rounded to 2 decimals.
- `recommendations` — `"Learn {skill}"` for missing core skills (blockers) and `"Consider {skill}"` for missing advanced ones (nice-to-haves) — the verb encodes priority.

**Expected (verified by running):** with the sample resume targeting `ml_engineer`, the output is `Coverage: 50%`, `Core met: ['Python', 'TensorFlow', 'Docker']`, `Core gaps: []`, `Advanced gaps: ['Kubernetes', 'MLOps', 'CI/CD']`, and three "Consider" recommendations. The core list is fully met — the resume is job-ready on the essentials — so every recommendation is a `Consider ...` item; had a core skill been missing it would have been promoted to `Learn ...`. Note `coverage` counts core and advanced equally: a candidate missing every core skill but knowing all advanced ones still scores 50%, which is why the met/gap lists matter more than the single number.

In [ ]:
class SkillGapAnalyzer:
    def __init__(self):
        self.skills_taxonomy = {
            "programming": ["Python", "Java", "C++", "JavaScript", "TypeScript", "Go", "Rust"],
            "ml_dl": ["TensorFlow", "PyTorch", "scikit-learn", "Keras", "XGBoost"],
            "nlp": ["NLP", "spaCy", "NLTK", "Transformers", "BERT", "GPT"],
            "data": ["SQL", "Pandas", "Spark", "Hadoop", "Tableau", "Power BI"],
            "cloud_devops": ["AWS", "Azure", "GCP", "Docker", "Kubernetes", "Terraform", "Jenkins"],
            "databases": ["PostgreSQL", "MySQL", "MongoDB", "Redis", "Elasticsearch"],
        }
        # Prerequisite map for career paths
        self.career_paths = {
            "data_scientist": {"core": ["Python", "SQL", "scikit-learn"], "advanced": ["TensorFlow", "PyTorch", "Spark"]},
            "ml_engineer": {"core": ["Python", "TensorFlow", "Docker"], "advanced": ["Kubernetes", "MLOps", "CI/CD"]},
            "nlp_engineer": {"core": ["Python", "NLP", "Transformers"], "advanced": ["PyTorch", "BERT", "spaCy"]},
        }
    
    def analyze_gaps(self, resume_text, target_role):
        resume_lower = resume_text.lower()
        found_skills = set()
        for cat, skills in self.skills_taxonomy.items():
            for skill in skills:
                if skill.lower() in resume_lower:
                    found_skills.add(skill)
        
        path = self.career_paths.get(target_role)
        if not path:
            return {"error": f"Unknown role: {target_role}"}
        
        core_found = [s for s in path["core"] if s in found_skills]
        core_missing = [s for s in path["core"] if s not in found_skills]
        adv_found = [s for s in path["advanced"] if s in found_skills]
        adv_missing = [s for s in path["advanced"] if s not in found_skills]
        
        coverage = len(core_found + adv_found) / max(len(path["core"] + path["advanced"]), 1)
        
        return {
            "role": target_role,
            "coverage": round(coverage, 2),
            "core_met": core_found,
            "core_gaps": core_missing,
            "advanced_met": adv_found,
            "advanced_gaps": adv_missing,
            "recommendations": [f"Learn {s}" for s in core_missing] + [f"Consider {s}" for s in adv_missing],
        }

resume = """Python developer with NLP and TensorFlow knowledge.
Used Docker for deployment."""
analyzer = SkillGapAnalyzer()
result = analyzer.analyze_gaps(resume, "ml_engineer")
print(f"Role: {result['role']}")
print(f"Coverage: {result['coverage']*100:.0f}%")
print(f"Core met: {result['core_met']}")
print(f"Core gaps: {result['core_gaps']}")
print(f"Advanced gaps: {result['advanced_gaps']}")
print("Recommendations:")
for r in result['recommendations']: print(f"  - {r}")

## Summary: Skill gap analysis identifies specific missing skills and career path recommendations.

**Know the gap, not just the score — per-skill comparison turns a 50% coverage number into a learning plan.**

By diffing found skills against a role's core/advanced lists, `analyze_gaps()` produces actionable output: which skills to `Learn` (core blockers) versus `Consider` (advanced optional), plus the raw met/gap lists for any custom logic downstream. The taxonomy constraint is the honest trade-off — detection is only as good as the curated skill list, and anything outside it is invisible.

This chapter feeds Ch. 54, where per-candidate skill data is aggregated into a single ranking across a whole applicant pool.